# Cleaning Dataset

Nievethan Thanienayagam

100750591

Dataset: https://github.com/hudl/open-data

# OnTarget Data Preprocessing & Feature Engineering Pipeline
**Dataset:** [Hudl / StatsBomb Open Data](https://github.com/hudl/open-data)

Nievethan Thaninayagam (100750591)

Gitae Yang ()

Arno Nazarian ()

## 1. Overview & Pitch Coordinate System
StatsBomb records match events in deeply nested JSON files, where each file (`data/events/<match_id>.json`) contains every on-ball action from a single match. To train our **OnTarget** and **Expected Goals (xG)** models, we need to filter these events down to shots, flatten the nested hierarchy, and engineer spatial features.

### Standardized Pitch Coordinates
StatsBomb normalizes all coordinates onto a **120 x 80 yard pitch** so that the attacking team always shoots from left to right toward $x = 120$:
* **Top-Left Corner:** $(0, 0)$
* **Bottom-Right Corner:** $(120, 80)$
* **Goal Center:** $(120, 40)$
* **Left Goalpost:** $(120, 36)$
* **Right Goalpost:** $(120, 44)$
* **Goal Width:** $8\text{ yards}$ (spanning $y \in [36, 44]$)

> **Prerequisites:** Make sure you have cloned the repository locally (`git clone https://github.com/hudl/open-data.git`) in your working directory so we can stream the JSON files directly from disk without hitting GitHub API rate limits.

<a href="https://colab.research.google.com/github/Nievethan/OnTarget/blob/main/DataPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import json
import math
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [12]:
import json
import math
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Point to your local clone of https://github.com/hudl/open-data
DATA_DIR = Path("./open-data-master/data")
EVENTS_DIR = DATA_DIR / "events"

# Pitch constants (StatsBomb standard 120x80 yard coordinate system)
GOAL_X = 120.0
GOAL_Y_CENTER = 40.0
GOAL_Y_LEFT = 36.0
GOAL_Y_RIGHT = 44.0
GOAL_WIDTH = 8.0


## 2. Spatial Geometry & Freeze-Frame Feature Engineering
Raw $(x, y)$ coordinates alone do not linearly capture how difficult a shot is. A shot from $(108, 40)$ is straight in front of the goal, whereas $(108, 20)$ is from a tight wide angle. We engineer two core geometric features and four defensive tracking features:

### A. Distance to Goal
The Euclidean distance from the shooter at $(x_s, y_s)$ to the center of the goal at $(120, 40)$:
$$d = \sqrt{(120 - x_s)^2 + (y_s - 40)^2}$$

### B. Visible Goal Angle ($\theta$)
The angle subtended by the two goalposts $(120, 36)$ and $(120, 44)$ from the shooter's perspective. Let $\Delta x = 120 - x_s$, $\Delta y = y_s - 40$, and goal width $w = 8$:
$$\theta = \text{atan2}\left(w \cdot \Delta x, \; \Delta x^2 + \Delta y^2 - \left(\frac{w}{2}\right)^2\right)$$
A larger angle means the shooter sees more of the net; an angle near $0^\circ$ means the shooter is almost on the touchline/byline.

### C. The "Shooting Cone" & Freeze-Frame Defenders
For every shot, StatsBomb records a `freeze_frame`: a snapshot of the $(x, y)$ locations of all visible teammates and opponents at the exact moment the ball is struck.
* **Shooting Cone:** The triangle formed by the shooter $(x_s, y_s)$, the left post $(120, 36)$, and the right post $(120, 44)$.
* Using a 2D cross-product (half-plane) sign test, we check every opponent (`teammate == False`) in the freeze frame to count how many field defenders (`defenders_in_cone`) and whether the goalkeeper (`gk_in_cone`) are directly blocking the path to the goal.

By mapping every `match_id` to it's competition metadata, we can analyze and extract the all the relevant shot data and fields to produce a lightweight dataset for sharing with the team.

In [13]:
def calculate_shot_geometry(x: float, y: float) -> Dict[str, float]:
    """Calculates Euclidean distance and visible goal angle (in radians and degrees)."""
    dx = max(GOAL_X - x, 1e-5)  # Prevent division by zero right on the goal line
    dy = y - GOAL_Y_CENTER
    distance = math.hypot(dx, dy)

    # Angle subtended by the two goalposts (120, 36) and (120, 44)
    angle_rad = math.atan2(
        GOAL_WIDTH * dx,
        dx**2 + dy**2 - (GOAL_WIDTH / 2.0) ** 2
    )
    if angle_rad < 0:
        angle_rad += math.pi

    return {
        "distance_to_goal": round(distance, 4),
        "shot_angle_rad": round(angle_rad, 4),
        "shot_angle_deg": round(math.degrees(angle_rad), 4),
    }


def is_point_in_shooting_cone(px: float, py: float, sx: float, sy: float) -> bool:
    """
    Checks if a player at (px, py) lies inside the triangle formed by
    Shooter (sx, sy), Left Post (120, 36), and Right Post (120, 44).
    """
    ax, ay = sx, sy
    bx, by = GOAL_X, GOAL_Y_LEFT
    cx, cy = GOAL_X, GOAL_Y_RIGHT

    # 2D cross-product sign check (barycentric half-plane test)
    d1 = (px - bx) * (ay - by) - (ax - bx) * (py - by)
    d2 = (px - cx) * (by - cy) - (bx - cx) * (py - cy)
    d3 = (px - ax) * (cy - ay) - (cx - ax) * (py - ay)

    has_neg = (d1 < 0) or (d2 < 0) or (d3 < 0)
    has_pos = (d1 > 0) or (d2 > 0) or (d3 > 0)

    return not (has_neg and has_pos)


def parse_freeze_frame(
    freeze_frame: Optional[List[Dict[str, Any]]], sx: float, sy: float
) -> Dict[str, Any]:
    """Extracts defensive pressure and goalkeeper positioning from the shot freeze frame."""
    if not freeze_frame:
        return {
            "has_freeze_frame": 0,
            "defenders_in_cone": np.nan,
            "gk_in_cone": np.nan,
            "closest_defender_dist": np.nan,
            "gk_distance_to_goal": np.nan,
        }

    defenders_in_cone = 0
    gk_in_cone = 0
    min_def_dist = float("inf")
    gk_dist_to_goal = np.nan

    for player in freeze_frame:
        if player.get("teammate", False):
            continue  # Skip teammates; only evaluate opposing players

        loc = player.get("location")
        if not loc or len(loc) < 2:
            continue

        px, py = float(loc[0]), float(loc[1])
        pos_name = player.get("position", {}).get("name", "")
        is_gk = pos_name == "Goalkeeper"
        in_cone = is_point_in_shooting_cone(px, py, sx, sy)

        if is_gk:
            gk_in_cone = int(in_cone)
            gk_dist_to_goal = round(math.hypot(GOAL_X - px, GOAL_Y_CENTER - py), 4)
        else:
            if in_cone:
                defenders_in_cone += 1
            dist_to_shooter = math.hypot(px - sx, py - sy)
            if dist_to_shooter < min_def_dist:
                min_def_dist = dist_to_shooter

    return {
        "has_freeze_frame": 1,
        "defenders_in_cone": defenders_in_cone,
        "gk_in_cone": gk_in_cone,
        "closest_defender_dist": round(min_def_dist, 4) if min_def_dist != float("inf") else np.nan,
        "gk_distance_to_goal": gk_dist_to_goal,
    }

## 3. Parsing & Flattening Event JSON Files
When extracting shots from StatsBomb event files, we handle three critical schema behaviors:

1. **Sparse Boolean Attributes:** In StatsBomb JSONs, boolean keys like `under_pressure`, `first_time`, `one_on_one`, and `open_goal` are **only present when `True`**. If a shot was *not* first-time, the key `"first_time"` is completely omitted from the JSON object. We use `.get(key, False)` and cast to `int` (`1` or `0`) so no `NaN` values are introduced.
2. **Filtering Penalties & Shootouts:**
   * `period == 5` represents a post-match Penalty Shootout (which does not count toward match stats).
   * `shot.type.name == 'Penalty'` represents a regular-time penalty kick. Because every penalty is taken from a static spot at $(108, 40)$ with no defenders in the cone and an ~76% conversion rate, mixing penalties into an open-play spatial model distorts distance and angle weights. We exclude them by default.
3. **Target Variable Definitions (`is_on_target` vs. `is_goal`):**
   * **`is_on_target = 1`** if `shot.outcome.name` is in `{"Goal", "Saved", "Saved to Post"}` (shots that would have entered the net without goalkeeper intervention).
   * **`is_on_target = 0`** if `shot.outcome.name` is in `{"Off T", "Blocked", "Post", "Wayward", "Saved Off Target"}`.
   * **`is_goal = 1`** strictly when `shot.outcome.name == "Goal"`.

In [14]:
ON_TARGET_OUTCOMES = {"Goal", "Saved", "Saved to Post"}

def extract_shots_from_events(
    event_files: List[Path], exclude_penalties: bool = True
) -> pd.DataFrame:
    """Streams event JSON files and extracts flattened shot records."""
    records = []

    for file_path in tqdm(event_files, desc="Parsing Event Files"):
        match_id = int(file_path.stem)
        with open(file_path, "r", encoding="utf-8") as f:
            events = json.load(f)

        for ev in events:
            # 1. Keep only Shot events outside of Penalty Shootouts (period 5)
            if ev.get("type", {}).get("name") != "Shot":
                continue
            if ev.get("period", 1) == 5:
                continue

            shot = ev.get("shot", {})
            shot_type = shot.get("type", {}).get("name", "Unknown")

            # 2. Exclude regular-time penalties so static 12-yard kicks don't skew geometry
            if exclude_penalties and shot_type == "Penalty":
                continue

            loc = ev.get("location")
            if not loc or len(loc) < 2:
                continue
            sx, sy = float(loc[0]), float(loc[1])

            # 3. Parse shot end location (x, y, and optional z height at the goal plane)
            end_loc = shot.get("end_location", [np.nan, np.nan, np.nan])
            end_x = float(end_loc[0]) if len(end_loc) >= 1 else np.nan
            end_y = float(end_loc[1]) if len(end_loc) >= 2 else np.nan
            end_z = float(end_loc[2]) if len(end_loc) >= 3 else np.nan

            outcome = shot.get("outcome", {}).get("name", "Unknown")

            # 4. Compute geometry & freeze-frame metrics
            geo = calculate_shot_geometry(sx, sy)
            ff_metrics = parse_freeze_frame(shot.get("freeze_frame"), sx, sy)

            # 5. Append flattened row dictionary
            records.append({
                "shot_id": ev.get("id"),
                "match_id": match_id,
                "period": ev.get("period"),
                "minute": ev.get("minute"),
                "second": ev.get("second"),
                "team_name": ev.get("team", {}).get("name"),
                "player_name": ev.get("player", {}).get("name"),
                # Spatial Coordinates & Geometry
                "x": sx,
                "y": sy,
                "end_x": end_x,
                "end_y": end_y,
                "end_z": end_z,
                **geo,
                # Categorical Attributes
                "play_pattern": ev.get("play_pattern", {}).get("name", "Unknown"),
                "shot_type": shot_type,
                "body_part": shot.get("body_part", {}).get("name", "Unknown"),
                "technique": shot.get("technique", {}).get("name", "Unknown"),
                # Boolean Context Flags (Missing in JSON = False)
                "under_pressure": int(ev.get("under_pressure", False)),
                "first_time": int(shot.get("first_time", False)),
                "one_on_one": int(shot.get("one_on_one", False)),
                "aerial_won": int(shot.get("aerial_won", False)),
                "open_goal": int(shot.get("open_goal", False)),
                "deflected": int(shot.get("deflected", False)),
                # Freeze-Frame Features
                **ff_metrics,
                # Benchmark & Targets
                "statsbomb_xg": shot.get("statsbomb_xg", np.nan),
                "outcome_raw": outcome,
                "is_on_target": int(outcome in ON_TARGET_OUTCOMES),
                "is_goal": int(outcome == "Goal"),
            })

    return pd.DataFrame(records)

# Discover all match JSON files (tip: slice [:500] for a quick test run)
all_event_files = sorted(list(EVENTS_DIR.glob("*.json")))
print(f"Found {len(all_event_files):,} match event files.")

df_raw = extract_shots_from_events(all_event_files, exclude_penalties=True)
print(f"Extracted {len(df_raw):,} raw non-penalty shots.")
df_raw.head()

Found 4,235 match event files.


Parsing Event Files:   0%|          | 0/4235 [00:00<?, ?it/s]

Extracted 106,724 raw non-penalty shots.


,shot_id,match_id,period,minute,second,team_name,player_name,x,y,end_x,...,deflected,has_freeze_frame,defenders_in_cone,gk_in_cone,closest_defender_dist,gk_distance_to_goal,statsbomb_xg,outcome_raw,is_on_target,is_goal
0,becd7956-ce44-479e-8fc9-16a2d1f1f349,15946,1,2,29,Barcelona,Lionel Andrés Messi Cuccittini,111.5,52.9,120.0,...,0,1,0,1,1.2166,6.6038,0.076992,Off T,0,0
1,9107d374-2942-4876-a14f-1b9f86901c15,15946,1,5,39,Barcelona,Jordi Alba Ramos,113.9,26.4,120.0,...,0,1,0,1,4.9769,3.5114,0.051668,Off T,0,0
2,ddd194ca-08fb-43d0-87c2-33647f975f9f,15946,1,15,29,Barcelona,Lionel Andrés Messi Cuccittini,93.7,34.7,117.5,...,0,1,1,1,1.6763,2.6401,0.016932,Saved,1,0
3,86596ddb-d824-4e5e-b18c-b4442e9ce7cf,15946,1,16,20,Deportivo Alavés,Rubén Sobrino Pozuelo,109.2,39.1,120.0,...,0,1,0,1,0.9055,2.6000,0.122604,Off T,0,0
4,3ed2b107-be17-42d5-9d1b-25006a0e55cb,15946,1,18,16,Barcelona,Luis Alberto Suárez Díaz,107.8,24.7,120.0,...,0,1,0,1,2.7731,3.5384,0.041751,Off T,0,0


## 4. Data Cleaning, Imputation & One-Hot Encoding
Before feeding the dataset into a machine learning model, we apply five cleaning rules:

1. **Deduplication:** Drop any duplicate `shot_id` rows (in case match files overlap).
2. **Coordinate Sanity Bounds:** Ensure $x \in [0, 120]$ and $y \in [0, 80]$ to remove rare tracking glitches.
3. **Missing Freeze Frames:** A small subset of historical matches lack freeze-frame tracking (`has_freeze_frame == 0`). Dropping those rows guarantees our defensive features (`defenders_in_cone`, `gk_in_cone`) are grounded in real tracking data rather than synthetic guesses.
4. **Edge-Case Defender Imputation:** Occasionally, a freeze frame exists (e.g., on a fast breakaway), but no field defenders were inside the camera frame, leaving `closest_defender_dist` as `NaN`. In that scenario, we impute `closest_defender_dist = 30.0` yards (ample open space) and `gk_distance_to_goal = 2.5` yards (standard goal-line positioning).
5. **One-Hot Encoding:** Convert categorical variables (`body_part`, `technique`, `play_pattern`, `shot_type`) into binary indicator columns (`0` or `1`) so linear models, neural networks, and tree ensembles can consume them directly.

In [15]:
def clean_and_encode_shots(df: pd.DataFrame, drop_missing_ff: bool = True) -> pd.DataFrame:
    df_clean = df.copy()

    # 1. Remove duplicate shot IDs if any exist
    df_clean = df_clean.drop_duplicates(subset=["shot_id"]).reset_index(drop=True)

    # 2. Enforce valid 120x80 pitch coordinate bounds
    valid_coords = (
        (df_clean["x"] >= 0.0) & (df_clean["x"] <= 120.0) &
        (df_clean["y"] >= 0.0) & (df_clean["y"] <= 80.0)
    )
    df_clean = df_clean[valid_coords].copy()

    # 3. Filter out shots without freeze-frame tracking data
    if drop_missing_ff:
        df_clean = df_clean[df_clean["has_freeze_frame"] == 1].copy()

    # 4. Impute edge-case missing defensive metrics (e.g., breakaways with no defenders in frame)
    df_clean["defenders_in_cone"] = df_clean["defenders_in_cone"].fillna(0).astype(int)
    df_clean["gk_in_cone"] = df_clean["gk_in_cone"].fillna(0).astype(int)
    df_clean["closest_defender_dist"] = df_clean["closest_defender_dist"].fillna(30.0)
    df_clean["gk_distance_to_goal"] = df_clean["gk_distance_to_goal"].fillna(2.5)

    # 5. One-Hot Encode categorical attributes
    categorical_cols = ["body_part", "technique", "play_pattern", "shot_type"]
    df_encoded = pd.get_dummies(
        df_clean,
        columns=categorical_cols,
        prefix=categorical_cols,
        drop_first=False,
        dtype=int
    )

    return df_encoded

df_clean = clean_and_encode_shots(df_raw, drop_missing_ff=True)
print(f"Cleaned dataset shape: {df_clean.shape}")

Cleaned dataset shape: (106717, 54)


## 5. Sanity Checks & Exploratory Verification
Before exporting the dataset, let's verify:
1. **Zero Missing Values:** Confirm no `NaN` values remain in our modeling feature columns.
2. **Target Class Balance:** Check the overall percentage of shots that are **On Target** (~33–35%) and result in a **Goal** (~9–11% for non-penalty shots).
3. **Outcome Breakdown:** Inspect how raw StatsBomb outcomes map to `is_on_target` and `is_goal`.

In [16]:
# Identify metadata/target columns vs. model feature columns
METADATA_AND_TARGETS = {
    "shot_id", "match_id", "team_name", "player_name",
    "end_x", "end_y", "end_z", "outcome_raw",
    "statsbomb_xg", "is_on_target", "is_goal"
}
feature_cols = [c for c in df_clean.columns if c not in METADATA_AND_TARGETS]

print(f"Total Modeling Features: {len(feature_cols)}")
print(f"Total Nulls in Features: {df_clean[feature_cols].isna().sum().sum()}")
print("-" * 50)
print(f"On-Target Rate (is_on_target == 1): {df_clean['is_on_target'].mean():.2%}")
print(f"Goal Conversion Rate (is_goal == 1): {df_clean['is_goal'].mean():.2%}")
print(f"Mean StatsBomb Benchmark xG:        {df_clean['statsbomb_xg'].mean():.4f}")
print("-" * 50)

# Verify target mapping against raw outcome strings
outcome_summary = (
    df_clean.groupby("outcome_raw")
    .agg(
        total_shots=("shot_id", "count"),
        is_on_target=("is_on_target", "first"),
        is_goal=("is_goal", "first"),
        avg_distance=("distance_to_goal", "mean"),
        avg_defenders_in_cone=("defenders_in_cone", "mean"),
    )
    .sort_values(by="total_shots", ascending=False)
)
display(outcome_summary)

Total Modeling Features: 43
Total Nulls in Features: 0
--------------------------------------------------
On-Target Rate (is_on_target == 1): 34.41%
Goal Conversion Rate (is_goal == 1): 10.24%
Mean StatsBomb Benchmark xG:        0.0966
--------------------------------------------------


,total_shots,is_on_target,is_goal,avg_distance,avg_defenders_in_cone
outcome_raw,,,,,
Off T,34440,0,0,20.210060,0.750842
Blocked,26706,0,0,20.358669,1.302217
Saved,25442,1,0,19.563722,0.535178
Goal,10925,1,1,13.010372,0.479268
Wayward,6063,0,0,16.704519,0.879103
Post,2310,0,0,16.876065,0.713420
Saved Off Target,477,0,0,24.516531,0.702306
Saved to Post,354,1,0,18.051486,0.655367


## 6. Exporting the Cleaned Dataset
We export the processed DataFrame to **Apache Parquet** (`ontarget_shots_cleaned.parquet`) rather than CSV because:
* It preserves exact column data types (`int64`, `float64`, `string`).
* It compresses the file to roughly 20% of the equivalent CSV size.
* Teammates can load it into their training notebooks in milliseconds via `pd.read_parquet("ontarget_shots_cleaned.parquet")` without re-running the JSON parser.

In [17]:
OUTPUT_PATH = Path("./ontarget_shots_cleaned.parquet")
df_clean.to_parquet(OUTPUT_PATH, index=False)
print(f"Successfully exported {len(df_clean):,} cleaned shots to: {OUTPUT_PATH.resolve()}")

Successfully exported 106,717 cleaned shots to: C:\Users\nieve\Desktop\OnTarget\ontarget_shots_cleaned.parquet
